# 142 — Simulación, sim-to-real y digital twins

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=142)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Especialista vs robusta

- (a) `E = (0.04/W)·0.95 + (1 − 0.04/W)·0.40 = 0.40 + 0.022/W`.
- (b) Conviene el especialista cuando `0.40 + 0.022/W > 0.88` ⇒
  `W < 0.0458`: solo si conoces μ con precisión de ±0.023 — prácticamente una
  medición de laboratorio.
- (c) W es tu **incertidumbre de identificación**: cada medición del hardware
  la estrecha. La fórmula convierte "cuánto instrumentar" en una decisión
  cuantitativa, que es exactamente la combinación identificación +
  randomización fina del README.


In [ ]:
for W in (0.5, 0.1, 0.05, 0.04, 0.02):
    e = 0.40 + 0.022/W
    mejor = "especialista" if e > 0.88 else "robusta"
    print(f"W={W}: E_esp={min(e, 0.95):.3f} -> {mejor}")


## Solución 2 — Gap simulado

Con μ=0.42: la especialista aplica 6.0 y necesita `|6.0 − 4.2| < 1` — falso
siempre: **0 %** de éxito (el 40 % del README venía de variabilidad extra; el
modelo simplificado lo lleva al extremo). La robusta percibe μ con ruido σ=0.05
⇒ error de fuerza ~N(0, 0.5), y `P(|error| < 1) ≈ 95 %`. Con μ uniforme en
(0.3, 0.8): la especialista solo acierta cuando μ cae cerca de 0.6
(`|6 − 10μ| < 1 ⇒ μ ∈ (0.5, 0.7)`, un 40 % del rango); la robusta mantiene
~95 % en todo el rango. La corrección basada en feedback vence a la constante
memorizada.


In [ ]:
import random
random.seed(142)

def exito(f, mu):
    return abs(f - mu*10) < 1

for escenario in ("fijo", "uniforme"):
    esp = rob = 0
    for _ in range(1000):
        mu = 0.42 if escenario == "fijo" else random.uniform(0.3, 0.8)
        esp += exito(6.0, mu)
        rob += exito((mu + random.gauss(0, 0.05))*10, mu)
    print(f"mu {escenario}: especialista={esp/10:.1f}% robusta={rob/10:.1f}%")


## Solución 3 — Componentes del gap

- (a) Reflejos no renderizados: **perceptivo**.
- (b) Latencia de motor 30 ms: **dinámico**.
- (c) Recompensa mal diseñada: **ninguno** — es un error de especificación
  que fallaría igual en sim; el gap no es el culpable de todo.
- (d) Fricción dependiente de humedad: **dinámico** (y además variable en el
  tiempo: candidato ideal a randomización).
- (e) Holgura del gripper: **dinámico** (dinámica no modelada).


## Solución 4 — Protocolo de 20 intentos

Un protocolo defendible: 5 intentos en la variante más fácil (validar que no
hay fallo catastrófico: si hay colisión dura o 0/5, parar y volver a sim),
luego 15 en condiciones nominales variando lo incontrolable (posición inicial,
iluminación). Criterio de parada: cualquier acción fuera de la envolvente de
fuerza/velocidad aborta la campaña. Con 14/20 = 70 %: la regla rápida da
incertidumbre ~±1/√20 ≈ ±22 puntos ⇒ el éxito real está plausiblemente entre
~48 % y ~92 %. Conclusión honesta: la política funciona pero el 92 % de sim
NO está confirmado; 20 intentos solo distinguen "funciona a grandes rasgos"
de "no funciona" — y eso ya justifica (o no) la siguiente campaña de pruebas.


In [ ]:
n, k = 20, 14
p = k/n
margen = 1/(n**0.5)
print(f"exito observado={p:.0%}, rango plausible ~[{p-margen:.0%}, {p+margen:.0%}]")
